# 1) MLP: A * B % C (A,B between 0..99, C between 0..19)

In [1]:
#@title (0) Setup
from google.colab import drive
import sys
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/Colab Notebooks')

%run "/content/drive/MyDrive/Colab Notebooks/Build your own LLM/han-common.ipynb"

Mounted at /content/drive
Train: 152,000 samples  |  Test: 38,000 samples
Input dim: 219  |  Output classes: 19
Label distribution (train): {np.int64(0): np.int64(51411), np.int64(1): np.int64(11977), np.int64(2): np.int64(14363), np.int64(3): np.int64(10413), np.int64(4): np.int64(11615), np.int64(5): np.int64(6717), np.int64(6): np.int64(10053), np.int64(7): np.int64(4823), np.int64(8): np.int64(6869), np.int64(9): np.int64(4650), np.int64(10): np.int64(4769), np.int64(11): np.int64(2542), np.int64(12): np.int64(4452), np.int64(13): np.int64(1747), np.int64(14): np.int64(2023), np.int64(15): np.int64(1374), np.int64(16): np.int64(1254), np.int64(17): np.int64(538), np.int64(18): np.int64(410)}
DNN(
  (proj_in): Linear(in_features=219, out_features=128, bias=True)
  (layers): ModuleList(
    (0-15): 16 x Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (norms): Modu

## 1.2) DNN: 219->512->512->256->256->128 w/ ReLU / TanH

In [2]:
class DNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, output_dim=OUTPUT_DIM, useTanh_m=False):
        super().__init__()
        act = nn.Tanh if useTanh_m else nn.ReLU
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),  act(),
            nn.Linear(512, 512),        act(),
            nn.Linear(512, 256),        act(),
            nn.Linear(256, 256),        act(),
            nn.Linear(256, 128),        act(),
            nn.Linear(128, output_dim),
        )

    def forward(self, x):
        return self.net(x)

MODEL = DNN().to(DEVICE)
print(MODEL)

print(f"\nTotal parameters: {sum(p.numel() for p in MODEL.parameters()):,}")
print(f"\nPer-layer shapes:")
for name, param in MODEL.named_parameters():
    if "weight" in name:
        print(f"  {name:30s}  {str(list(param.shape)):>15s}  ({param.numel():>6,} params)")


DNN(
  (net): Sequential(
    (0): Linear(in_features=219, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): ReLU()
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): ReLU()
    (10): Linear(in_features=128, out_features=19, bias=True)
  )
)

Total parameters: 607,763

Per-layer shapes:
  net.0.weight                         [512, 219]  (112,128 params)
  net.2.weight                         [512, 512]  (262,144 params)
  net.4.weight                         [256, 512]  (131,072 params)
  net.6.weight                         [256, 256]  (65,536 params)
  net.8.weight                         [128, 256]  (32,768 params)
  net.10.weight                         [19, 128]  ( 2,432 params)


## 1.3) Train

With 5 hidden layers and 7K samples A*B % 13 is much harder than A+B % 5.

In [3]:
# ── Hyperparameters ──
LR = 1e-3
WEIGHT_DECAY = 0.1
MAX_EPOCHS = 100

def init_weights(m, method):
    if isinstance(m, nn.Linear):
        if method == "uniform":
            nn.init.uniform_(m.weight, -1, 1)
        elif method == "he":
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        elif method == "xavier":
            nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def train(init="default", run_name="default", useTanh=False):
    model = DNN(useTanh_m=useTanh).to(DEVICE)
    if init != "default":
        _init = lambda m: init_weights(m, init)
    else:
        _init = None

    return train_common(model, run_name, LR, WEIGHT_DECAY, MAX_EPOCHS, _init)

# 2) Uniform distribution [-1, 1] + ReLU:  won't converge

In [ ]:
train(init="uniform", run_name="Uniform+ReLU", useTanh=False, )

  🚀 Run: Uniform+ReLU


/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


Epoch      1 | Train loss 3.3723  acc 33.5% | Test loss 3.4074  acc 33.3%
Epoch      2 | Train loss 2.5302  acc 33.8% | Test loss 2.5625  acc 33.6%
Epoch      3 | Train loss 2.4087  acc 33.8% | Test loss 2.4361  acc 33.6%
Epoch      4 | Train loss 2.3734  acc 33.8% | Test loss 2.3948  acc 33.6%
Epoch      5 | Train loss 2.3614  acc 33.8% | Test loss 2.3780  acc 33.6%
Epoch      6 | Train loss 2.3562  acc 33.8% | Test loss 2.3692  acc 33.6%
Epoch      7 | Train loss 2.3532  acc 33.8% | Test loss 2.3640  acc 33.6%
Epoch      8 | Train loss 2.3498  acc 33.8% | Test loss 2.3593  acc 33.6%
Epoch      9 | Train loss 2.3333  acc 33.8% | Test loss 2.3406  acc 33.6%
Epoch     10 | Train loss 2.3005  acc 33.8% | Test loss 2.3085  acc 33.6%
Epoch     11 | Train loss 2.2473  acc 33.8% | Test loss 2.2569  acc 33.6%
Epoch     12 | Train loss 2.2071  acc 33.8% | Test loss 2.2170  acc 33.6%
Epoch     13 | Train loss 2.1609  acc 33.8% | Test loss 2.1709  acc 33.6%
Epoch     14 | Train loss 2.1364  acc 

# 3) He/Kaiming + ReLU - converges!


In [ ]:
train(init="he", run_name="He+ReLU", useTanh=False )

  🚀 Run: He+ReLU
Epoch      1 | Train loss 1.0558  acc 54.9% | Test loss 1.0661  acc 54.0%
Epoch      2 | Train loss 1.0266  acc 55.2% | Test loss 1.0374  acc 53.8%
Epoch      3 | Train loss 1.0233  acc 55.5% | Test loss 1.0339  acc 54.1%
Epoch      4 | Train loss 0.9273  acc 60.6% | Test loss 0.9367  acc 59.9%
Epoch      5 | Train loss 0.8428  acc 63.7% | Test loss 0.8514  acc 62.7%
Epoch      6 | Train loss 0.7568  acc 67.8% | Test loss 0.7630  acc 66.7%
Epoch      7 | Train loss 0.7245  acc 68.2% | Test loss 0.7312  acc 67.4%
Epoch      8 | Train loss 0.7157  acc 68.8% | Test loss 0.7255  acc 67.3%
Epoch      9 | Train loss 0.6904  acc 70.1% | Test loss 0.7004  acc 69.3%
Epoch     10 | Train loss 0.6672  acc 70.7% | Test loss 0.6788  acc 69.8%
Epoch     11 | Train loss 0.5896  acc 72.9% | Test loss 0.5972  acc 72.4%
Epoch     12 | Train loss 0.4757  acc 79.3% | Test loss 0.4836  acc 78.6%
Epoch     13 | Train loss 0.3939  acc 82.7% | Test loss 0.4008  acc 82.2%
Epoch     14 | Train 

# 4) Xavier + Tanh - converges but slowly

In [ ]:
#@title Answer
train(init="xavier", useTanh=True, run_name="Xavier+Tanh")

  🚀 Run: Xavier+Tanh
Epoch      1 | Train loss 1.1301  acc 53.6% | Test loss 1.1353  acc 53.2%
Epoch      2 | Train loss 1.0309  acc 54.7% | Test loss 1.0340  acc 54.5%
Epoch      3 | Train loss 1.0306  acc 54.8% | Test loss 1.0345  acc 54.2%
Epoch      4 | Train loss 1.0300  acc 54.7% | Test loss 1.0322  acc 54.4%
Epoch      5 | Train loss 1.0347  acc 54.8% | Test loss 1.0389  acc 54.3%
Epoch      6 | Train loss 0.9776  acc 59.0% | Test loss 0.9821  acc 58.4%
Epoch      7 | Train loss 0.9500  acc 60.2% | Test loss 0.9549  acc 59.6%
Epoch      8 | Train loss 0.9092  acc 60.8% | Test loss 0.9124  acc 60.5%
Epoch      9 | Train loss 0.9078  acc 60.9% | Test loss 0.9103  acc 60.7%
Epoch     10 | Train loss 0.9309  acc 60.5% | Test loss 0.9367  acc 60.1%
Epoch     11 | Train loss 0.9102  acc 61.1% | Test loss 0.9138  acc 60.7%
Epoch     12 | Train loss 0.8903  acc 61.6% | Test loss 0.8948  acc 61.1%
Epoch     13 | Train loss 0.8807  acc 61.7% | Test loss 0.8843  acc 61.5%
Epoch     14 | Tr

# 5) Xavier + ReLU: converges slower than He+ReLU

In [4]:
#@title Answer
train(init="xavier", run_name="Xavier+ReLU", useTanh=False)

  🚀 Run: Xavier+ReLU


W1024 17:10:40.783000 1938 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch      1 | Train loss 1.0658  acc 54.6% | Test loss 1.0736  acc 54.1%
Epoch      2 | Train loss 1.0325  acc 55.0% | Test loss 1.0403  acc 53.7%
Epoch      3 | Train loss 1.0049  acc 57.5% | Test loss 1.0170  acc 56.5%
Epoch      4 | Train loss 0.8649  acc 62.8% | Test loss 0.8734  acc 62.0%
Epoch      5 | Train loss 0.7918  acc 65.2% | Test loss 0.7987  acc 64.2%
Epoch      6 | Train loss 0.7361  acc 68.4% | Test loss 0.7429  acc 67.4%
Epoch      7 | Train loss 0.6439  acc 74.0% | Test loss 0.6492  acc 73.2%
Epoch      8 | Train loss 0.6043  acc 76.5% | Test loss 0.6109  acc 75.8%
Epoch      9 | Train loss 0.5326  acc 80.1% | Test loss 0.5366  acc 79.7%
Epoch     10 | Train loss 0.5055  acc 80.6% | Test loss 0.5122  acc 80.3%
Epoch     11 | Train loss 0.4388  acc 82.4% | Test loss 0.4396  acc 82.3%
Epoch     12 | Train loss 0.4136  acc 83.7% | Test loss 0.4180  acc 83.3%
Epoch     13 | Train loss 0.4152  acc 83.8% | Test loss 0.4205  acc 83.4%
Epoch     14 | Train loss 0.3815  acc 

OptimizedModule(
  (_orig_mod): DNN(
    (net): Sequential(
      (0): Linear(in_features=219, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): ReLU()
      (4): Linear(in_features=512, out_features=256, bias=True)
      (5): ReLU()
      (6): Linear(in_features=256, out_features=256, bias=True)
      (7): ReLU()
      (8): Linear(in_features=256, out_features=128, bias=True)
      (9): ReLU()
      (10): Linear(in_features=128, out_features=19, bias=True)
    )
  )
)

# 6) Comparison Across Runs

In [ ]:
#@title Visualize
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import logging

# Suppress missing font warnings
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# Prefer the deployment font when available, otherwise fall back to common sans-serif fonts
font_candidates = ['Inter', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
plt.rcParams['font.family'] = font_candidates
font_path = 'fonts/Inter-Regular.ttf'
try:
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
except Exception:
    pass

# Set styling for all text and lines
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.edgecolor'] = '#F1F5F9'

# Grid styling
plt.rcParams['grid.color'] = '#5B7268'
plt.rcParams['grid.alpha'] = 0.7

def _coplot_luminance_from_rgba(rgba):
    r, g, b = mcolors.to_rgb(rgba)
    def _linear(c):
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * _linear(r) + 0.7152 * _linear(g) + 0.0722 * _linear(b)

def _coplot_text_color_for_rgba(rgba, preferred='#F1F5F9'):
    try:
        preferred_l = _coplot_luminance_from_rgba(mcolors.to_rgba(preferred))
        bg_l = _coplot_luminance_from_rgba(rgba)
        lighter = max(preferred_l, bg_l)
        darker = min(preferred_l, bg_l)
        ratio = (lighter + 0.05) / (darker + 0.05)
        if ratio >= 4.5:
            return preferred
    except Exception:
        pass
    bg_l = _coplot_luminance_from_rgba(rgba)
    return '#111111' if bg_l > 0.5 else '#FFFFFF'

# Create figure and axis with publication-quality settings
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
fig.patch.set_facecolor('#1A3A2A')

# Distinct bright colors for dark background (modified to keep runs distinguishable)
colors = ["#38BDF8", "#34D399", "#FBBF24", "#F87171", "#A78BFA"]
line_styles = ["-","--",":","-."]
marker_styles = ["o","s","^","D"]

for idx, run in enumerate(all_runs):
    h = run["history"]

    line_style = line_styles[idx % len(line_styles)] if False else '-'
    marker_style = marker_styles[idx % len(marker_styles)] if False else ''

    # Plot accuracy
    axes[0].plot(h["epoch"], h["test_acc"], label=run["name"],
                 color=colors[idx % len(colors)], linewidth=2,
                 linestyle=line_style, marker=marker_style, alpha=0.9)

    # Plot loss
    axes[1].plot(h["epoch"], h["test_loss"], label=run["name"],
                 color=colors[idx % len(colors)], linewidth=2,
                 linestyle=line_style, marker=marker_style, alpha=0.9)

# Apply custom styling to both axes
for ax in axes:
    ax.set_facecolor('#1A3A2A')
    ax.set_xlabel('Epoch', fontsize=12, labelpad=10, color='#F1F5F9')

    # Add grid
    ax.grid(axis='both', alpha=0.7, color='#5B7268', linestyle='--', linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)

    # Remove top and right spines for cleaner look
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_color('#F1F5F9')
    ax.spines['left'].set_color('#F1F5F9')

    # Adjust spine positions
    ax.spines['left'].set_position(('outward', 2))
    ax.spines['bottom'].set_position(('outward', 2))

    # Add legend
    ax.legend(frameon=False, fontsize=10)

# Specific titles and labels
axes[0].set_ylabel('Test Accuracy', fontsize=12, color='#F1F5F9', labelpad=10)
axes[0].set_title('Test Accuracy by Init Method', color='#F1F5F9', pad=15)

axes[1].set_ylabel('Test Loss', fontsize=12, color='#F1F5F9', labelpad=10)
axes[1].set_title('Test Loss by Init Method', color='#F1F5F9', pad=15)
axes[1].set_yscale('log')

# Global title
plt.suptitle("Initialization Method Comparison", fontsize=14, y=1.02, color='#F1F5F9')

# Tight layout
plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Run':<20s} {'Converged Epoch':<20s}")
print("-" * 40)
for run in sorted(all_runs, key=lambda x: (x['converged_epoch'] is None, x['converged_epoch'])):
    ep = run["converged_epoch"]
    print(f"{run['name']:<20s} {str(ep) if ep else 'Did not converge':<20s}")



Run                  Converged Epoch     
----------------------------------------
Xavier+ReLU          34                  
He+ReLU              37                  
Uniform+ReLU         88                  
Xavier+Tanh          Did not converge    


In [5]:
#@title (7) Save and load model
FILENAME = "DDN-v2.pth"
MODEL = get_model('Xavier+ReLU')

print("--- Testing Local Save & Load ---")
load_model = DNN().to(DEVICE)
load_model = save_load_model_local(MODEL, load_model, filename_c=FILENAME)
predicted_value = invoke_model_with_a_b_c(load_model, 13, 1, 10)

--- Testing Local Save & Load ---
model weights saved locally to: .//DDN-v2.pth
model weights loaded successfully into a new model instance.
Prediction for (13 * 1) % 10 = 3


In [6]:
#@title (8) Save to Google Drive
pathname = save_to_google_drive(MODEL, FILENAME)

Model saved to Google Drive: /content/drive/MyDrive/workshop/DDN-v2.pth


In [7]:
#@title (9) Load Pretrained model from Hugging Face
print("\n--- Testing Hugging Face Download & Load ---")
hf_model = DNN().to(DEVICE)
hf_model = load_model_hugging_face(new_model_c=hf_model, filename_c=FILENAME)
predicted_value = invoke_model_with_a_b_c(hf_model, 12, 1, 10)

MODEL = hf_model


--- Testing Hugging Face Download & Load ---


DDN-v2.pth: reconstructing file:   0%|          |  0.00B / 2.43MB            

DDN-v2.pth: downloading bytes:           |  0.00B            

model downloaded from HF to: /root/.cache/huggingface/hub/models--howardyang2009--llm-workshop-v1/snapshots/c01a9309e30e876bfaebab7ee04d7d93b744e217/DDN-v2.pth
model weights loaded successfully into a new model instance.
Prediction for (12 * 1) % 10 = 2
